# Bybit — 1h Klines (OHLCV)

This notebook builds the hourly OHLCV dataset used in the empirical section. It pulls 1-hour candlesticks from Bybit’s REST API, reshapes them into a tidy time series, and saves both an analysis-ready table and a small set of traceability artifacts (raw payloads, QA summary, and a run manifest).

## Context
The empirical pipeline relies on several hourly series (prices, open interest, funding). This notebook produces the **price/volume leg** of that pipeline: a clean OHLCV panel indexed by an hourly timestamp in **UTC**, designed to be joined on `ts` with the other datasets.

## Source and time conventions
Data are retrieved from the Bybit REST API (market data / klines). All timestamps are handled in **UTC**. The extraction window follows the standard half-open interval convention **[START, END_EXCL)** so that adjacent runs can be concatenated without overlap.

## What you need to set (configuration cell)
- `SYMBOL` (e.g., `BTCUSDT`)
- `CATEGORY` (market type used in the study design, e.g., `linear`)
- `START` and `END_EXCL` (UTC; `END_EXCL` is exclusive)

The notebook fetches the series by paging through the requested interval and consolidating the responses into one hourly dataset.

## What the notebook does (in order)
1) **Parameter parsing and path setup**  
   Reads the inputs, defines the extraction window, and prepares the output locations under `data/`.

2) **API retrieval (paged requests)**  
   Calls the klines endpoint repeatedly to cover the full window. Requests are rate-limited and retried to reduce failures due to transient network/API issues.

3) **Normalization to a tidy table**  
   Converts the raw API format into a flat table with typed columns and a single hourly timestamp (`ts`, UTC). The table is sorted and de-duplicated on `(symbol, ts)`.

4) **Quality checks (QA)**  
   Computes basic integrity checks: time bounds, duplicates, missing hours (gaps in the hourly index), missing values, and simple OHLC consistency checks.

5) **Export and provenance**  
   Writes:
   - a raw JSONL file (audit trail),
   - a processed Parquet file (analysis-ready),
   - a QA JSON file (diagnostics),
   - a manifest JSON file (run parameters + SHA256 hashes of the outputs).

## Outputs
All outputs are written under `data/`:

- `data/raw/...jsonl` — raw API payloads (audit trail)
- `data/processed/...parquet` — normalized 1h OHLCV table
- `data/qa/..._qa.json` — QA summary (gaps, duplicates, bounds, missingness)
- `data/manifests/..._manifest.json` — parameters and SHA256 hashes

## Processed dataset (schema)
One row per hour (UTC):
- `ts` (datetime, UTC; candle start)
- `open`, `high`, `low`, `close`
- `volume`
- `turnover` (if provided by the API)
- `symbol`, `category`, `interval_min`, `source`, `ingested_at_utc`

## Notes on reproducibility
This notebook is fully reproducible as a procedure (same code + same parameters). Exact bit-for-bit reproducibility depends on the provider returning identical historical data for the same query window. The manifest hashes make it explicit which exact artifacts were produced in a given run.

## Quick notes
- If you plan multiple runs, keep one run per subfolder under `data/` (raw/processed/qa/manifests) to avoid overwriting.
- When something looks off, check the QA JSON first (missing hours, duplicates, time bounds), then confirm the window and symbol.

In [22]:
import json, time, hashlib, random
from pathlib import Path
from typing import Dict, Any, List, Tuple

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ===== CONFIG =====
OUT_DIR = Path("raw_data_lake") / "bybit_klines_1h"   # simple, dataset-centric
BASE_URL = "https://api.bybit.com"

CATEGORY = "linear"
SYMBOL = "ETHUSDT"
INTERVAL = "60"          # 1h on Bybit V5
LIMIT = 1000             # max 1000 :contentReference[oaicite:2]{index=2}

START = pd.Timestamp("2021-03-15T00:00:00Z")
END_EXCL = pd.Timestamp("2025-12-01T00:00:00Z")      # end exclusive

MIN_REQUEST_INTERVAL_S = 0.6
TIMEOUT = (10, 120)
RETRIES_TOTAL = 8
BACKOFF_FACTOR = 1.0

CHUNK_DAYS = 365
MAX_PAGES_PER_CHUNK = 20000

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR =", OUT_DIR)
print("Range:", START, "->", END_EXCL, "(end exclusive)")

OUT_DIR = raw_data_lake/bybit_klines_1h
Range: 2021-03-15 00:00:00+00:00 -> 2025-12-01 00:00:00+00:00 (end exclusive)


In [2]:
def ms(ts: pd.Timestamp) -> int:
    return int(ts.value // 10**6)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def append_jsonl(path: Path, obj: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def utc_hour_grid(start: pd.Timestamp, end_excl: pd.Timestamp) -> pd.DatetimeIndex:
    return pd.date_range(start=start, end=end_excl, freq="h", inclusive="left", tz="UTC")

def now_utc_iso() -> str:
    return pd.Timestamp.utcnow().isoformat()


In [4]:
def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=RETRIES_TOTAL,
        backoff_factor=BACKOFF_FACTOR,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s

SESSION = make_session()
_last_req_t = 0.0

def _pace():
    global _last_req_t
    now = time.monotonic()
    wait = MIN_REQUEST_INTERVAL_S - (now - _last_req_t)
    if wait > 0:
        time.sleep(wait + random.uniform(0.0, 0.15))
    _last_req_t = time.monotonic()

def bybit_get(path: str, params: Dict[str, Any]) -> Dict[str, Any]:
    url = f"{BASE_URL}{path}"

    for attempt in range(1, RETRIES_TOTAL + 1):
        _pace()
        t0 = time.time()
        r = SESSION.get(url, params=params, timeout=TIMEOUT)
        elapsed = time.time() - t0

        try:
            data = r.json()
        except Exception:
            data = {"retCode": None, "retMsg": "Non-JSON response", "_http_status": r.status_code}

        # rate limit
        if data.get("retCode") == 10006:
            sleep_s = min(60.0, (2 ** (attempt - 1)) * 0.8 + random.uniform(0.0, 0.8))
            print(f"[rate_limit] 10006 attempt={attempt}/{RETRIES_TOTAL} sleep={sleep_s:.2f}s")
            time.sleep(sleep_s)
            continue

        return {"elapsed_s": elapsed, "response": data, "http_status": r.status_code, "url": url}

    raise RuntimeError("Rate limit persists after retries.")


In [7]:
def fetch_klines_1h_raw(start: pd.Timestamp, end_excl: pd.Timestamp) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    pages_path = OUT_DIR / "pages.jsonl"
    if pages_path.exists():
        pages_path.unlink()

    all_rows: List[list] = []
    requests_made = stalls = empty_pages = 0

    chunk_start = start
    while chunk_start < end_excl:
        chunk_end = min(chunk_start + pd.Timedelta(days=CHUNK_DAYS), end_excl)
        print(f"[kline] chunk {chunk_start:%Y-%m-%d} -> {chunk_end:%Y-%m-%d}")

        cursor_end = chunk_end
        prev_sig = None
        safety = 0

        while True:
            safety += 1
            if safety > MAX_PAGES_PER_CHUNK:
                raise RuntimeError("Too many pages in chunk; reduce CHUNK_DAYS.")

            params = {
                "category": CATEGORY,
                "symbol": SYMBOL,
                "interval": INTERVAL,
                "start": ms(chunk_start),
                "end": ms(cursor_end),
                "limit": LIMIT,
            }

            page = bybit_get("/v5/market/kline", params)
            res = page["response"]
            requests_made += 1

            append_jsonl(pages_path, {
                "t": time.time(),
                "path": "/v5/market/kline",
                "params": params,
                "http_status": page["http_status"],
                "elapsed_s": page["elapsed_s"],
                "retCode": res.get("retCode"),
                "retMsg": res.get("retMsg"),
            })

            if res.get("retCode") != 0:
                raise RuntimeError(f"Bybit retCode={res.get('retCode')} retMsg={res.get('retMsg')}")

            lst = (res.get("result") or {}).get("list") or []
            if not lst:
                empty_pages += 1
                break

            ts = [int(r[0]) for r in lst]
            page_min_ms, page_max_ms = min(ts), max(ts)
            sig = (len(lst), page_min_ms, page_max_ms)
            if prev_sig is not None and sig == prev_sig:
                stalls += 1
                print("[kline][warn] repeated page signature -> break chunk")
                break
            prev_sig = sig

            all_rows.extend(lst)

            page_min = pd.to_datetime(page_min_ms, unit="ms", utc=True)
            if page_min <= chunk_start:
                break

            new_cursor_end = page_min - pd.Timedelta(milliseconds=1)
            if new_cursor_end >= cursor_end:
                stalls += 1
                print("[kline][warn] cursor not moving -> break chunk")
                break
            cursor_end = new_cursor_end

        chunk_start = chunk_end

    df = pd.DataFrame(all_rows, columns=["timestamp_ms","open","high","low","close","volume","turnover"])

    # dedup by timestamp_ms
    df["timestamp_ms"] = pd.to_numeric(df["timestamp_ms"], errors="coerce")
    df = df.dropna(subset=["timestamp_ms"])
    df["timestamp_ms"] = df["timestamp_ms"].astype("int64")
    df = df.drop_duplicates("timestamp_ms", keep="last").sort_values("timestamp_ms").reset_index(drop=True)

    summary = {
        "requests_made": requests_made,
        "rows_raw": int(len(all_rows)),
        "rows_dedup": int(len(df)),
        "stalls": stalls,
        "empty_pages": empty_pages,
    }
    return df, summary


In [9]:
def normalize_klines_to_hourly(df_raw: pd.DataFrame, start: pd.Timestamp, end_excl: pd.Timestamp):
    df = df_raw.copy()
    df["date"] = pd.to_datetime(df["timestamp_ms"], unit="ms", utc=True).dt.floor("h")

    for c in ["open","high","low","close","volume","turnover"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.sort_values("date").drop_duplicates("date", keep="last")
    df = df[(df["date"] >= start) & (df["date"] < end_excl)].copy()

    idx = utc_hour_grid(start, end_excl)
    out = pd.DataFrame({"date": idx}).merge(
        df[["date","open","high","low","close","volume","turnover"]],
        on="date", how="left"
    )

    qa = {
        "expected_hours": int(len(idx)),
        "rows_after_dedup": int(len(df)),
        "missing_close": int(out["close"].isna().sum()),
        "missing_first_20": [str(x) for x in out.loc[out["close"].isna(), "date"].head(20).tolist()],
    }
    return out, qa


In [11]:
df_raw, http_summary = fetch_klines_1h_raw(START, END_EXCL)
df_norm, qa = normalize_klines_to_hourly(df_raw, START, END_EXCL)

raw_csv = OUT_DIR / "klines_raw.csv"
raw_parq = OUT_DIR / "klines_raw.parquet"
norm_parq = OUT_DIR / "klines_1h.parquet"
qa_path = OUT_DIR / "qa_report.json"

df_raw.to_csv(raw_csv, index=False)
df_raw.to_parquet(raw_parq, index=False, engine="pyarrow")

df_norm.to_parquet(norm_parq, index=False, engine="pyarrow")
qa_path.write_text(json.dumps(qa, indent=2), encoding="utf-8")

manifest = {
    "dataset": "bybit_klines_1h",
    "collected_at_utc": now_utc_iso(),
    "config": {
        "base_url": BASE_URL,
        "category": CATEGORY,
        "symbol": SYMBOL,
        "interval": INTERVAL,
        "limit": LIMIT,
        "start": str(START),
        "end_excl": str(END_EXCL),
        "chunk_days": CHUNK_DAYS,
        "min_request_interval_s": MIN_REQUEST_INTERVAL_S,
        "timeout": TIMEOUT,
        "retries_total": RETRIES_TOTAL,
        "backoff_factor": BACKOFF_FACTOR,
    },
    "files": {
        "raw_csv": str(raw_csv),
        "raw_parquet": str(raw_parq),
        "norm_parquet": str(norm_parq),
        "pages_jsonl": str(OUT_DIR / "pages.jsonl"),
        "qa_report": str(qa_path),
    },
    "hashes": {
        "raw_csv_sha256": sha256_file(raw_csv),
        "raw_parquet_sha256": sha256_file(raw_parq),
        "norm_parquet_sha256": sha256_file(norm_parq),
        "qa_report_sha256": sha256_file(qa_path),
    },
    "http_summary": http_summary,
    "qa": qa,
}

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("✅ Saved:", norm_parq)
print("HTTP summary:", http_summary)
print("QA:", qa)

df_norm.head()


[kline] chunk 2021-03-15 -> 2022-03-15
[kline] chunk 2022-03-15 -> 2023-03-15
[kline] chunk 2023-03-15 -> 2024-03-14
[kline] chunk 2024-03-14 -> 2025-03-14
[kline] chunk 2025-03-14 -> 2025-12-01
✅ Saved: raw_data_lake/bybit_klines_1h/klines_1h.parquet
HTTP summary: {'requests_made': 43, 'rows_raw': 41333, 'rows_dedup': 41329, 'stalls': 0, 'empty_pages': 0}
QA: {'expected_hours': 41328, 'rows_after_dedup': 41328, 'missing_close': 0, 'missing_first_20': []}


,date,open,high,low,close,volume,turnover
0,2021-03-15 00:00:00+00:00,1850.35,1885.0,1842.60,1875.00,4436.19,8.317856e+06
1,2021-03-15 01:00:00+00:00,1875.00,1894.0,1870.00,1887.30,1946.20,3.673063e+06
2,2021-03-15 02:00:00+00:00,1887.30,1894.3,1882.40,1885.20,1978.25,3.729397e+06
3,2021-03-15 03:00:00+00:00,1885.20,1892.2,1885.20,1886.55,1540.79,2.906777e+06
4,2021-03-15 04:00:00+00:00,1886.55,1893.5,1858.85,1864.05,4940.45,9.209246e+06
